### Import modules

In [138]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


### Import Data

In [76]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
data['pattern'].fillna('None', inplace=True)

synthetic_data       = data[data['file'].str.contains('pattern',na=False)]
verified_communities = data[~data['file'].str.contains('pattern',na=False)]

/tmp/ipykernel_83284/859644980.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['pattern'].fillna('None', inplace=True)


### Custom Train Test Split

In [91]:
def get_folded_splits(fold_count=4,random_state=42):
    folded_data = []

    _sd = synthetic_data.copy()
    _vd = verified_communities.copy()

    _vd_grouped = _vd.groupby('pattern')

    _sd_counts = _sd['pattern'].value_counts()
    _vd_counts = _vd['pattern'].value_counts()

    verified_training_data = pd.DataFrame()

    for label, vd_group in _vd_grouped:
        total_count = _sd_counts.get(label,0) + _vd_counts.get(label,0)
        max_train_count = int(total_count/fold_count)

        print(f"Label: {label}, Total Count: {total_count}, Max Train Count: {max_train_count}, VD Count: {_vd_counts.get(label,0)}")

        vd_count = len(vd_group)
        selected_indexes = np.random.choice(vd_group.index,size=min(vd_count, max_train_count),replace=False)
        verified_training_data = pd.concat([verified_training_data, vd_group.loc[selected_indexes]])
        _vd = _vd.drop(index=selected_indexes)

    kfold = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=random_state)
    folds = kfold.split(_sd, _sd['pattern'])

    for train_indices, test_indices in folds:
        sd_train_data = _sd.iloc[train_indices]
        sd_test_data  = _sd.iloc[test_indices]
        train_data = pd.concat([sd_train_data, verified_training_data])
        test_data  = pd.concat([sd_test_data, _vd])
            
        folded_data.append((train_data, test_data))
    
    return folded_data


### Preprocess

In [ ]:
TARGET_COLUMN = 'pattern'

In [77]:
def prepare_data(data):
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)

    X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
        X,
        y_encoded,
        test_size=0.25,
        random_state=42,
        stratify=y_encoded,
    )

    return X_train, X_temp, y_train_enc, y_temp_enc, label_encoder, num_classes

In [103]:
def preprocess_data_nn(data):
    X_train, X_temp, y_train_enc, y_temp_enc, label_encoder, num_classes = prepare_data(data)

    X_val, X_test, y_val_enc, y_test_enc = train_test_split(
        X_temp,
        y_temp_enc,
        test_size=0.5,
        random_state=42,
        stratify=y_temp_enc,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
    y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
    y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)
    return X_train, X_val, X_test, y_train, y_val, y_test,y_test_enc, label_encoder, num_classes, scaler

### NN Model

In [81]:
def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            Dense(num_classes, activation="softmax"),
        ]
    )

In [142]:
kfolded_data = get_folded_splits(fold_count=4)
for fold in kfolded_data:
    print(fold[0].shape,fold[1].shape)

Label: Advanced LLM Prompting, Total Count: 105, Max Train Count: 26, VD Count: 8
Label: Classical Models, Total Count: 116, Max Train Count: 29, VD Count: 47
Label: Enhanced User Intent Comprehension with LLMs, Total Count: 98, Max Train Count: 24, VD Count: 8
Label: Integrating External Knlowladge with LLM, Total Count: 101, Max Train Count: 25, VD Count: 4
Label: LLM Agent Training & Alignment, Total Count: 98, Max Train Count: 24, VD Count: 4
Label: LLM Context Management, Total Count: 97, Max Train Count: 24, VD Count: 2
Label: LLM Results Evaluation, Total Count: 100, Max Train Count: 25, VD Count: 13
Label: LLM based Multimodal Generative Prompting, Total Count: 102, Max Train Count: 25, VD Count: 26
Label: LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT, Total Count: 98, Max Train Count: 24, VD Count: 8
Label: Model Abstraction Pattern, Total Count: 99, Max Train Count: 24, VD Count: 16
Label: Modular LLM Agent Architectures, Total C

In [143]:
fold_results = []
for fold_index, (train_data, test_data) in enumerate(kfolded_data):
    print(f"Processing Fold {fold_index + 1}")

    X_train, X_val, X_test, y_train, y_val, y_test,y_test_enc, label_encoder, num_classes, scaler = preprocess_data_nn(train_data)


    model = build_classifier(X_train.shape[1], num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
    )

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=callbacks,
        verbose=0,
    )

    test_loss, test_acc, test_top3 = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

    y_pred = model.predict(X_test)
    y_pred_labels = y_pred.argmax(axis=1)
    report = classification_report(
        y_test_enc,
        y_pred_labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    fold_results.append({
        "summary": summary,
        "class_breakdown": class_breakdown,
        "history": history.history,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_top3": test_top3
    })

Processing Fold 1
Test loss: 1.6157 | Test accuracy: 0.6976 | Test top-3 accuracy: 0.8341
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Processing Fold 2
Test loss: 1.1210 | Test accuracy: 0.8098 | Test top-3 accuracy: 0.9220
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Processing Fold 3
Test loss: 1.5494 | Test accuracy: 0.7171 | Test top-3 accuracy: 0.8683
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Processing Fold 4
Test loss: 1.1382 | Test accuracy: 0.7902 | Test top-3 accuracy: 0.8976
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


In [144]:
print("Average Accuracy:", round(float(np.mean([float(fold_result['summary'][f'f1-score']['accuracy']) for i, fold_result in enumerate(fold_results)])), 3))

Average Accuracy: 0.754


### Logistic Regression

In [145]:
kfolded_data = get_folded_splits(fold_count=4)

fold_results = []
for fold_index, (train_data, test_data) in enumerate(kfolded_data):
    print(f"Processing Fold {fold_index + 1}")
    
    X_train, X_test, y_train_enc, y_test_enc, label_encoder, num_classes = prepare_data(data)
    logreg = LogisticRegression(max_iter=1000,n_jobs=-1)
    logreg.fit(X_train, y_train_enc)
    y_pred = logreg.predict(X_test)
    report = classification_report(
        y_test_enc,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    fold_results.append({
        "summary": summary,
        "class_breakdown": class_breakdown,
    })

Label: Advanced LLM Prompting, Total Count: 105, Max Train Count: 26, VD Count: 8
Label: Classical Models, Total Count: 116, Max Train Count: 29, VD Count: 47
Label: Enhanced User Intent Comprehension with LLMs, Total Count: 98, Max Train Count: 24, VD Count: 8
Label: Integrating External Knlowladge with LLM, Total Count: 101, Max Train Count: 25, VD Count: 4
Label: LLM Agent Training & Alignment, Total Count: 98, Max Train Count: 24, VD Count: 4
Label: LLM Context Management, Total Count: 97, Max Train Count: 24, VD Count: 2
Label: LLM Results Evaluation, Total Count: 100, Max Train Count: 25, VD Count: 13
Label: LLM based Multimodal Generative Prompting, Total Count: 102, Max Train Count: 25, VD Count: 26
Label: LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT, Total Count: 98, Max Train Count: 24, VD Count: 8
Label: Model Abstraction Pattern, Total Count: 99, Max Train Count: 24, VD Count: 16
Label: Modular LLM Agent Architectures, Total C

In [146]:
print("Average Accuracy:", round(float(np.mean([float(fold_result['summary'][f'f1-score']['accuracy']) for i, fold_result in enumerate(fold_results)])), 3))

Average Accuracy: 0.657
